In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r'full_data_2.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 966 entries, 0 to 965
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      966 non-null    int64  
 1   Name            966 non-null    str    
 2   Price           337 non-null    float64
 3   Screen Size     966 non-null    float64
 4   Display         966 non-null    int64  
 5   Chipset         966 non-null    int64  
 6   NFC             966 non-null    int64  
 7   ROM             966 non-null    float64
 8   RAM             966 non-null    float64
 9   Battery         966 non-null    float64
 10  Refresh Rate    966 non-null    float64
 11  Brand           966 non-null    str    
 12  antutu_11       747 non-null    float64
 13  clock           966 non-null    float64
 14  gpu             747 non-null    str    
 15  total_cores     966 non-null    float64
 16  min_freq        966 non-null    float64
 17  mean_freq       966 non-null    float64
 18  O

In [3]:
def compute_correlation(df, target, method: str = "pearson", min_samples: int = 30, exclude_cols: list = None):
    if target not in df.columns:
        raise ValueError(f"Cột '{target}' không tồn tại trong DataFrame.")
 
    exclude = set(exclude_cols or []) | {target}
    numeric_cols = [c for c in df.select_dtypes(include="number").columns
                    if c not in exclude]
 
    target_series = df[target]
    records = []
 
    for col in numeric_cols:
        pair = df[[col, target]].dropna()
        n = len(pair)
        if n < min_samples:
            continue
 
        x, y = pair[col].values, pair[target].values
 
        if method == "pearson":
            r, p = stats.pearsonr(x, y)
        elif method == "spearman":
            r, p = stats.spearmanr(x, y)
        elif method == "kendall":
            r, p = stats.kendalltau(x, y)
        else:
            raise ValueError("method phải là 'pearson', 'spearman', hoặc 'kendall'")
 
        abs_r = abs(r)
        if   abs_r >= 0.7: strength = "rất mạnh"
        elif abs_r >= 0.5: strength = "mạnh"
        elif abs_r >= 0.3: strength = "trung bình"
        elif abs_r >= 0.1: strength = "yếu"
        else:              strength = "rất yếu"
 
        records.append({
            "feature":     col,
            "correlation": round(r, 4),
            "p_value":     round(p, 6),
            "n_samples":   n,
            "strength":    strength,
        })
 
    result = (pd.DataFrame(records)
                .assign(abs_corr=lambda d: d["correlation"].abs())
                .sort_values("abs_corr", ascending=False)
                .drop(columns="abs_corr")
                .reset_index(drop=True))
    return result

In [4]:
result = compute_correlation(df, 'antutu_11')
result

,feature,correlation,p_value,n_samples,strength
0,mean_freq,0.9428,0.000000,747,rất mạnh
1,clock,0.9200,0.000000,747,rất mạnh
2,min_freq,0.8166,0.000000,747,rất mạnh
3,Price,0.7539,0.000000,254,rất mạnh
4,RAM,0.5600,0.000000,747,mạnh
5,ROM,0.5134,0.000000,747,mạnh
6,SIM_total,0.4899,0.000000,747,trung bình
7,has_eSIM,0.4674,0.000000,747,trung bình
8,PPI,0.4553,0.000000,747,trung bình
9,Display,0.4415,0.000000,747,trung bình


PREDICT AND IMPUTE

In [5]:
FEATURES = [
    "perf_freq_ghz", "eff_freq_ghz", "perf_cores", "eff_cores",
    "gpu_family_enc", "Chipset", "Display", "RAM", "ROM",
    "NFC", "PPI", "has_eSIM", "SIM_total", "OS_Version",
    "Battery", "Screen Size", "Refresh Rate",
]
TARGET = "antutu_11"

In [6]:
#best parameter
GBM_PARAMS = dict(
    n_estimators    = 300,
    max_depth       = 3,
    learning_rate   = 0.1,
    subsample       = 0.7,
    min_samples_leaf= 5,
    random_state    = 42,
)

In [7]:
df_train = df[df[TARGET].notna()].copy()
df_miss  = df[df[TARGET].isna()].copy()

In [8]:
print(f"Train (có antutu_11: {len(df_train)}")
print(f"Missing (cần impute: {len(df_miss)}")

Train (có antutu_11: 747
Missing (cần impute: 219


In [9]:
X_train = df_train[FEATURES]
y_train = df_train[TARGET]
X_miss = df_miss[FEATURES]

KeyError: "['perf_freq_ghz', 'eff_freq_ghz', 'perf_cores', 'eff_cores', 'OS_Version'] not in index"

In [ ]:
print("...Cross_validation (5-fold) trên tập train...")

...Cross_validation (5-fold) trên tập train...


In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy = "median")),
    ("model", GradientBoostingRegressor(**GBM_PARAMS)),
])

In [ ]:
cv = KFold(n_splits = 5, shuffle = True, random_state = 42)

In [ ]:
r2_scores = cross_val_score(pipeline, X_train, y_train, cv = cv, scoring = 'r2')
rmse_scores = np.sqrt(-cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="neg_mean_squared_error"))

In [ ]:
print(f"R²: {r2_scores.mean():.4f}  ± {r2_scores.std():.4f}")
print(f"RMSE: {rmse_scores.mean():,.0f}  ± {rmse_scores.std():,.0f}")
print(f"RMSE % of mean: {rmse_scores.mean() / y_train.mean() * 100:.1f}%")

R²: 0.7863  ± 0.0777
RMSE: 376,996  ± 68,126
RMSE % of mean: 35.9%


In [ ]:
print("...Train model trên toàn bộ tập train...")
pipeline.fit(X_train, y_train)
print("Done.")

...Train model trên toàn bộ tập train...
Done.


In [ ]:
print("...Impute antutu_11 cho các dòng thiếu...")
predicted = pipeline.predict(X_miss)
predicted = np.clip(predicted, 0, None)
df.loc[df[TARGET].isna(), TARGET] = predicted

...Impute antutu_11 cho các dòng thiếu...


In [ ]:
df["antutu_11_imputed"] = False
df.loc[df_miss.index, "antutu_11_imputed"] = True
 
print(f"Imputed {len(df_miss)} giá trị.")
print(f"Predicted range : {predicted.min():,.0f} — {predicted.max():,.0f}")
print(f"Predicted mean  : {predicted.mean():,.0f}")
print(f"Original mean   : {y_train.mean():,.0f}")

Imputed 219 giá trị.
Predicted range : 0 — 3,512,749
Predicted mean  : 755,135
Original mean   : 1,051,576


In [ ]:
df[df['antutu_11_imputed'] == True]['antutu_11']

14     1.410864e+06
16     1.766314e+06
24     7.622398e+05
26     4.742659e+05
30     6.152830e+05
           ...     
930    5.476079e+05
931    5.370118e+05
934    1.781760e+06
948    1.329738e+06
961    8.741550e+05
Name: antutu_11, Length: 219, dtype: float64